## RAG -Tradinitional


### DATA Ingestion - Vector DB PipeLine

In [2]:
from langchain_core.documents import Document

In [3]:
#pdf loader
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_9157/616835706.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [5]:
def process_documents(dir_path):
    "load all the pdf files"
    all_pdfs=[]
    path=Path(dir_path)
    pdf_files=list(path.glob("**/*.pdf"))
    for pdf_file in pdf_files:
        print(f"{pdf_file.name} processing")
        try:
            loader=PyPDFLoader(pdf_file)
            pdfs = loader.load()
            for pdf in pdfs:
                pdf.metadata["source"]=pdf_file.name
                pdf.metadata["typr"]="pdf"
            all_pdfs.extend(pdfs)
            print(f"loaded {len(pdfs)}")
        except Exception as e:
            print(f"Error: {e}")
    return all_pdfs 

all_documents=process_documents("/home/sampath/Documents/ langchain-learning/TRAG/data")


Data_visualization_U-2.pdf processing
loaded 26
UNIT-4.pdf processing
loaded 16


In [6]:
all_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-02T20:42:27+05:30', 'author': 'Giri Babu Kande', 'moddate': '2026-01-02T20:42:27+05:30', 'source': 'Data_visualization_U-2.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'typr': 'pdf'}, page_content='1 \n \nUNIT-2: Human Perception and Information Processing, What Is Perception , \nPhysiology, Perceptual Processing, Perception in Visualization, Metrics, Cogni-\ntion , Visualization Foundations, The Visualization Process in Detail, Semiology \nof Graphical Symbols, The Eight Visual, Variables Historical Perspective , \nTaxonomies  \n  \n \nHuman Perception and Information Processing \nHuman perception and information processing  play a crucial role in data \nvisualization because visualizations are effective only when they match how \nhumans see, interpret, and understand visual information. The goal of \nvisualization is to reduce cognitive effort and help users q

In [13]:
def split_pdfs(all_documents,chunk_size=2000,chunk_overlap=200):
    "recursively split the documents into chunks"
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_documents=text_splitter.split_documents(all_documents)
    return split_documents

In [14]:
chunks=split_pdfs(all_documents)
chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-01-02T20:42:27+05:30', 'author': 'Giri Babu Kande', 'moddate': '2026-01-02T20:42:27+05:30', 'source': 'Data_visualization_U-2.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'typr': 'pdf'}, page_content='1 \n \nUNIT-2: Human Perception and Information Processing, What Is Perception , \nPhysiology, Perceptual Processing, Perception in Visualization, Metrics, Cogni-\ntion , Visualization Foundations, The Visualization Process in Detail, Semiology \nof Graphical Symbols, The Eight Visual, Variables Historical Perspective , \nTaxonomies  \n  \n \nHuman Perception and Information Processing \nHuman perception and information processing  play a crucial role in data \nvisualization because visualizations are effective only when they match how \nhumans see, interpret, and understand visual information. The goal of \nvisualization is to reduce cognitive effort and help users q

## Embedding and VectorStoreDb

In [18]:
import numpy as np 
from sentence_transformers import SentenceTransformer
from typing import Dict,Any,List,Tuple
import uuid
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity